# Collaborative Filtering Interview Prep: From Implementation to Evaluation

This notebook demonstrates a production-ready implementation of collaborative filtering recommendation systems, designed for ML engineering interviews.


## 1. Theory & Design Overview

### What is Collaborative Filtering?

Collaborative Filtering (CF) is a recommendation technique that predicts user preferences by leveraging the behavior of similar users or items. The core assumption is **homophily**: users with similar past behavior will have similar future preferences.

### User-based vs Item-based CF

**User-based CF:**
- Finds users similar to the target user
- Predicts ratings based on how similar users rated the item
- Formula: $\hat{r}_{ui} = \bar{r}_u + \frac{\sum_{v \in N(u)} sim(u,v) \cdot (r_{vi} - \bar{r}_v)}{\sum_{v \in N(u)} |sim(u,v)|}$

**Item-based CF:**
- Finds items similar to the target item
- Predicts ratings based on how the user rated similar items
- Formula: $\hat{r}_{ui} = \bar{r}_i + \frac{\sum_{j \in N(i)} sim(i,j) \cdot r_{uj}}{\sum_{j \in N(i)} |sim(i,j)|}$

### Explicit vs Implicit Feedback

- **Explicit**: Direct ratings (1-5 stars, thumbs up/down)
- **Implicit**: Indirect signals (clicks, views, purchases, time spent)

### Why CF Works

1. **Homophily**: Similar users have similar tastes
2. **Co-occurrence**: Items frequently consumed together are similar
3. **Sparsity**: Most user-item pairs are unobserved (sparse matrix)
4. **Cold Start**: New users/items lack interaction history

### Key Interview Assumptions

- **Sparse matrix**: Most entries are missing (0 or NaN)
- **Cold start exists**: New users/items need special handling
- **Offline evaluation**: Train/test split on historical data


In [ ]:
import numpy as np
from typing import Optional, Tuple
import warnings
warnings.filterwarnings('ignore')


## 3. Similarity Utilities (Shared Code)

### Cosine Similarity

Cosine similarity measures the angle between two vectors, independent of magnitude:
$$sim(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{||\mathbf{u}|| \cdot ||\mathbf{v}||}$$

**Why cosine similarity?**
- Handles sparse vectors well (ignores missing values in dot product)
- Normalized to [-1, 1] range
- Computationally efficient
- Time complexity: O(n² · d) for n vectors of dimension d


In [ ]:
def cosine_similarity_matrix(X: np.ndarray) -> np.ndarray:
    """
    Compute pairwise cosine similarity matrix for rows of X.
    
    Args:
        X: (n, d) array where each row is a vector
        
    Returns:
        (n, n) similarity matrix
    """
    # Normalize each row to unit length
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1, norms)  # Avoid division by zero
    X_norm = X / norms
    
    # Cosine similarity = dot product of normalized vectors
    sim_matrix = np.dot(X_norm, X_norm.T)
    
    # Ensure diagonal is 1.0 (self-similarity)
    np.fill_diagonal(sim_matrix, 1.0)
    
    return sim_matrix


def safe_normalize(x: np.ndarray, axis: int = 0) -> np.ndarray:
    """
    Normalize array along axis, handling zero-norm vectors.
    
    Args:
        x: Input array
        axis: Axis to normalize along
        
    Returns:
        Normalized array (zero vectors remain zero)
    """
    norms = np.linalg.norm(x, axis=axis, keepdims=True)
    return np.where(norms > 0, x / norms, x)


def top_k_similar(sim_matrix: np.ndarray, k: int, exclude_self: bool = True) -> np.ndarray:
    """
    Find top-K most similar entities for each row.
    
    Args:
        sim_matrix: (n, n) similarity matrix
        k: Number of neighbors to return
        exclude_self: If True, exclude self-similarity (diagonal)
        
    Returns:
        (n, k) array of indices of top-K neighbors
    """
    n = sim_matrix.shape[0]
    k = min(k, n - (1 if exclude_self else 0))
    
    if exclude_self:
        # Mask diagonal to exclude self
        sim_matrix = sim_matrix.copy()
        np.fill_diagonal(sim_matrix, -np.inf)
    
    # Get top-K indices for each row
    top_k_indices = np.argsort(sim_matrix, axis=1)[:, -k:][:, ::-1]
    
    return top_k_indices


## 2. Collaborative Filtering Implementations

### Design Principles

- **Clean separation**: Model logic separate from data
- **Reusable**: Classes can be instantiated and reused
- **Testable**: Each method has clear inputs/outputs
- **Edge case handling**: Graceful handling of cold start, zero similarities


In [ ]:
class UserBasedCF:
    """
    User-based Collaborative Filtering.
    
    Predicts ratings by finding similar users and aggregating their ratings.
    """
    
    def __init__(self, k: int = 10, similarity: str = 'cosine'):
        """
        Args:
            k: Number of similar users to consider (top-K neighbors)
            similarity: Similarity metric ('cosine' only for now)
        """
        self.k = k
        self.similarity = similarity
        self.ratings_matrix = None  # (n_users, n_items)
        self.user_similarity = None  # (n_users, n_users)
        self.user_means = None  # (n_users,) mean rating per user
        
    def fit(self, ratings_matrix: np.ndarray):
        """
        Train the model on a user-item rating matrix.
        
        Args:
            ratings_matrix: (n_users, n_items) array
                           Missing ratings should be 0 or NaN
        """
        # Convert NaN to 0 for computation
        self.ratings_matrix = np.nan_to_num(ratings_matrix, nan=0.0)
        n_users, n_items = self.ratings_matrix.shape
        
        # Compute user means (for mean-centering)
        # Only count non-zero ratings
        user_counts = np.sum(self.ratings_matrix != 0, axis=1)
        user_sums = np.sum(self.ratings_matrix, axis=1)
        self.user_means = np.where(user_counts > 0, user_sums / user_counts, 0.0)
        
        # Mean-center the ratings matrix
        ratings_centered = self.ratings_matrix - self.user_means[:, np.newaxis]
        
        # Compute user-user similarity matrix
        if self.similarity == 'cosine':
            self.user_similarity = cosine_similarity_matrix(ratings_centered)
        else:
            raise ValueError(f"Similarity '{self.similarity}' not supported")
        
        return self
    
    def predict(self, user_id: int, item_id: int) -> float:
        """
        Predict rating for a specific user-item pair.
        
        Args:
            user_id: User index
            item_id: Item index
            
        Returns:
            Predicted rating
        """
        if self.ratings_matrix is None:
            raise ValueError("Model must be fitted before prediction")
        
        # Get top-K similar users
        top_k_users = top_k_similar(self.user_similarity, self.k)[user_id]
        
        # Get ratings from similar users for this item
        neighbor_ratings = self.ratings_matrix[top_k_users, item_id]
        neighbor_similarities = self.user_similarity[user_id, top_k_users]
        
        # Filter: only consider neighbors who rated this item
        valid_mask = neighbor_ratings != 0
        if not np.any(valid_mask):
            # Cold start: no similar users rated this item
            return self.user_means[user_id] if self.user_means[user_id] > 0 else 0.0
        
        neighbor_ratings = neighbor_ratings[valid_mask]
        neighbor_similarities = neighbor_similarities[valid_mask]
        
        # Mean-center neighbor ratings
        neighbor_means = self.user_means[top_k_users[valid_mask]]
        neighbor_ratings_centered = neighbor_ratings - neighbor_means
        
        # Weighted sum: sum(sim * rating) / sum(|sim|)
        similarity_sum = np.sum(np.abs(neighbor_similarities))
        if similarity_sum == 0:
            return self.user_means[user_id] if self.user_means[user_id] > 0 else 0.0
        
        prediction_centered = np.sum(neighbor_similarities * neighbor_ratings_centered) / similarity_sum
        prediction = self.user_means[user_id] + prediction_centered
        
        # Clip to reasonable range (assuming 1-5 scale)
        return np.clip(prediction, 0.0, 5.0)
    
    def predict_all(self) -> np.ndarray:
        """
        Predict ratings for all user-item pairs.
        
        Returns:
            (n_users, n_items) array of predicted ratings
        """
        n_users, n_items = self.ratings_matrix.shape
        predictions = np.zeros((n_users, n_items))
        
        for u in range(n_users):
            for i in range(n_items):
                predictions[u, i] = self.predict(u, i)
        
        return predictions


In [ ]:
class ItemBasedCF:
    """
    Item-based Collaborative Filtering.
    
    Predicts ratings by finding similar items and using user's ratings on those items.
    """
    
    def __init__(self, k: int = 10, similarity: str = 'cosine'):
        """
        Args:
            k: Number of similar items to consider (top-K neighbors)
            similarity: Similarity metric ('cosine' only for now)
        """
        self.k = k
        self.similarity = similarity
        self.ratings_matrix = None  # (n_users, n_items)
        self.item_similarity = None  # (n_items, n_items)
        self.item_means = None  # (n_items,) mean rating per item
        
    def fit(self, ratings_matrix: np.ndarray):
        """
        Train the model on a user-item rating matrix.
        
        Args:
            ratings_matrix: (n_users, n_items) array
                           Missing ratings should be 0 or NaN
        """
        # Convert NaN to 0 for computation
        self.ratings_matrix = np.nan_to_num(ratings_matrix, nan=0.0)
        n_users, n_items = self.ratings_matrix.shape
        
        # Compute item means (for mean-centering)
        item_counts = np.sum(self.ratings_matrix != 0, axis=0)
        item_sums = np.sum(self.ratings_matrix, axis=0)
        self.item_means = np.where(item_counts > 0, item_sums / item_counts, 0.0)
        
        # Mean-center the ratings matrix
        ratings_centered = self.ratings_matrix - self.item_means[np.newaxis, :]
        
        # Compute item-item similarity matrix
        # Transpose to get items as rows
        if self.similarity == 'cosine':
            self.item_similarity = cosine_similarity_matrix(ratings_centered.T)
        else:
            raise ValueError(f"Similarity '{self.similarity}' not supported")
        
        return self
    
    def predict(self, user_id: int, item_id: int) -> float:
        """
        Predict rating for a specific user-item pair.
        
        Args:
            user_id: User index
            item_id: Item index
            
        Returns:
            Predicted rating
        """
        if self.ratings_matrix is None:
            raise ValueError("Model must be fitted before prediction")
        
        # Get top-K similar items
        top_k_items = top_k_similar(self.item_similarity, self.k)[item_id]
        
        # Get user's ratings for similar items
        neighbor_ratings = self.ratings_matrix[user_id, top_k_items]
        neighbor_similarities = self.item_similarity[item_id, top_k_items]
        
        # Filter: only consider items the user rated
        valid_mask = neighbor_ratings != 0
        if not np.any(valid_mask):
            # Cold start: user hasn't rated similar items
            return self.item_means[item_id] if self.item_means[item_id] > 0 else 0.0
        
        neighbor_ratings = neighbor_ratings[valid_mask]
        neighbor_similarities = neighbor_similarities[valid_mask]
        
        # Mean-center neighbor ratings
        neighbor_means = self.item_means[top_k_items[valid_mask]]
        neighbor_ratings_centered = neighbor_ratings - neighbor_means
        
        # Weighted sum: sum(sim * rating) / sum(|sim|)
        similarity_sum = np.sum(np.abs(neighbor_similarities))
        if similarity_sum == 0:
            return self.item_means[item_id] if self.item_means[item_id] > 0 else 0.0
        
        prediction_centered = np.sum(neighbor_similarities * neighbor_ratings_centered) / similarity_sum
        prediction = self.item_means[item_id] + prediction_centered
        
        # Clip to reasonable range (assuming 1-5 scale)
        return np.clip(prediction, 0.0, 5.0)
    
    def predict_all(self) -> np.ndarray:
        """
        Predict ratings for all user-item pairs.
        
        Returns:
            (n_users, n_items) array of predicted ratings
        """
        n_users, n_items = self.ratings_matrix.shape
        predictions = np.zeros((n_users, n_items))
        
        for u in range(n_users):
            for i in range(n_items):
                predictions[u, i] = self.predict(u, i)
        
        return predictions


## 4. Prediction Logic

### Weighted Sum Formula

The core prediction formula for collaborative filtering:

$$\hat{r}_{ui} = \bar{r}_u + \frac{\sum_{v \in N(u)} sim(u,v) \cdot (r_{vi} - \bar{r}_v)}{\sum_{v \in N(u)} |sim(u,v)|}$$

**Key components:**
1. **Mean-centering**: Subtract user/item mean to handle rating bias
2. **Weighted sum**: Similarity-weighted average of neighbor ratings
3. **Normalization**: Divide by sum of absolute similarities to prevent scale issues

### User-based vs Item-based

- **User-based**: "Users similar to you liked this item"
- **Item-based**: "You liked items similar to this one"

**Item-based advantages:**
- More stable (items change less than users)
- Better scalability (fewer items than users typically)
- Interpretable ("if you liked X, you'll like Y")


## 5. Evaluation Metrics

### Rating Prediction Metrics (Explicit Feedback)

For explicit ratings (1-5 stars), we measure prediction accuracy:

- **RMSE** (Root Mean Squared Error): Penalizes large errors more
- **MAE** (Mean Absolute Error): Average prediction error

### Ranking Metrics (Implicit Feedback)

For implicit feedback (clicks, views), we care about ranking quality:

- **Precision@K**: Of top-K recommendations, how many are relevant?
- **Recall@K**: Of all relevant items, how many are in top-K?

**Why RMSE alone is insufficient:**
- Recommender systems are often evaluated on ranking, not just rating accuracy
- A small RMSE doesn't guarantee good recommendations
- Need to balance accuracy with diversity and novelty


In [ ]:
def rmse(y_true: np.ndarray, y_pred: np.ndarray, mask: Optional[np.ndarray] = None) -> float:
    """
    Compute Root Mean Squared Error.
    
    Args:
        y_true: True ratings
        y_pred: Predicted ratings
        mask: Optional boolean mask for valid entries
        
    Returns:
        RMSE value
    """
    if mask is not None:
        y_true = y_true[mask]
        y_pred = y_pred[mask]
    
    if len(y_true) == 0:
        return np.nan
    
    return np.sqrt(np.mean((y_true - y_pred) ** 2))


def mae(y_true: np.ndarray, y_pred: np.ndarray, mask: Optional[np.ndarray] = None) -> float:
    """
    Compute Mean Absolute Error.
    
    Args:
        y_true: True ratings
        y_pred: Predicted ratings
        mask: Optional boolean mask for valid entries
        
    Returns:
        MAE value
    """
    if mask is not None:
        y_true = y_true[mask]
        y_pred = y_pred[mask]
    
    if len(y_true) == 0:
        return np.nan
    
    return np.mean(np.abs(y_true - y_pred))


def precision_at_k(y_true: np.ndarray, y_pred: np.ndarray, k: int, threshold: float = 3.0) -> float:
    """
    Compute Precision@K.
    
    Args:
        y_true: True ratings (binary relevance: >= threshold is relevant)
        y_pred: Predicted ratings
        k: Number of top items to consider
        threshold: Rating threshold for relevance
        
    Returns:
        Precision@K value
    """
    # Get top-K predicted items
    top_k_indices = np.argsort(y_pred)[-k:][::-1]
    
    # Count relevant items in top-K
    relevant_in_topk = np.sum(y_true[top_k_indices] >= threshold)
    
    return relevant_in_topk / k if k > 0 else 0.0


def recall_at_k(y_true: np.ndarray, y_pred: np.ndarray, k: int, threshold: float = 3.0) -> float:
    """
    Compute Recall@K.
    
    Args:
        y_true: True ratings (binary relevance: >= threshold is relevant)
        y_pred: Predicted ratings
        k: Number of top items to consider
        threshold: Rating threshold for relevance
        
    Returns:
        Recall@K value
    """
    # Total relevant items
    total_relevant = np.sum(y_true >= threshold)
    
    if total_relevant == 0:
        return 0.0
    
    # Get top-K predicted items
    top_k_indices = np.argsort(y_pred)[-k:][::-1]
    
    # Count relevant items in top-K
    relevant_in_topk = np.sum(y_true[top_k_indices] >= threshold)
    
    return relevant_in_topk / total_relevant


## 6. Toy Dataset + Train/Test Split

### Creating a Synthetic User-Item Matrix

We'll create a small, interpretable dataset to demonstrate the algorithms:
- **Rows**: Users (5 users)
- **Columns**: Items (6 items)
- **Values**: Ratings on 1-5 scale
- **Missing values**: Represented as 0 (will be masked during evaluation)


In [ ]:
# Create a toy user-item rating matrix
# Rows = users, Columns = items
# Ratings on scale 1-5, 0 = missing/not rated

np.random.seed(42)

# Create base matrix with some structure
n_users, n_items = 5, 6
ratings_full = np.array([
    [5, 4, 0, 0, 3, 4],  # User 0: likes action/sci-fi
    [4, 5, 3, 0, 0, 4],  # User 1: likes action/sci-fi
    [0, 0, 5, 4, 5, 0],  # User 2: likes romance/comedy
    [0, 0, 4, 5, 4, 0],  # User 3: likes romance/comedy
    [3, 3, 3, 3, 3, 3],  # User 4: neutral, rates everything
])

print("Full Rating Matrix:")
print("Rows = Users (0-4), Columns = Items (0-5)")
print(ratings_full)
print(f"\nShape: {ratings_full.shape}")
print(f"Sparsity: {np.sum(ratings_full == 0) / ratings_full.size * 100:.1f}%")


Full Rating Matrix:
Rows = Users (0-4), Columns = Items (0-5)
[[5 4 0 0 3 4]
 [4 5 3 0 0 4]
 [0 0 5 4 5 0]
 [0 0 4 5 4 0]
 [3 3 3 3 3 3]]

Shape: (5, 6)
Sparsity: 33.3%


In [ ]:
def train_test_split(ratings_matrix: np.ndarray, test_ratio: float = 0.2, random_state: int = 42):
    """
    Split user-item matrix into train and test sets.
    
    Args:
        ratings_matrix: Full rating matrix
        test_ratio: Proportion of non-zero entries to use as test
        random_state: Random seed
        
    Returns:
        train_matrix, test_matrix, test_mask
    """
    np.random.seed(random_state)
    
    # Find all non-zero (observed) ratings
    observed_mask = ratings_matrix != 0
    observed_indices = np.argwhere(observed_mask)
    
    # Randomly select test set
    n_test = int(len(observed_indices) * test_ratio)
    test_indices = np.random.choice(len(observed_indices), size=n_test, replace=False)
    test_positions = observed_indices[test_indices]
    
    # Create test mask
    test_mask = np.zeros_like(ratings_matrix, dtype=bool)
    test_mask[test_positions[:, 0], test_positions[:, 1]] = True
    
    # Create train matrix (hide test ratings)
    train_matrix = ratings_matrix.copy()
    train_matrix[test_mask] = 0
    
    # Create test matrix (only test entries)
    test_matrix = np.zeros_like(ratings_matrix)
    test_matrix[test_mask] = ratings_matrix[test_mask]
    
    return train_matrix, test_matrix, test_mask


# Perform train/test split
train_matrix, test_matrix, test_mask = train_test_split(ratings_full, test_ratio=0.2)

print("Training Matrix (test ratings masked as 0):")
print(train_matrix)
print(f"\nTest entries: {np.sum(test_mask)}")
print("\nTest Matrix (only test entries shown):")
print(test_matrix)


Training Matrix (test ratings masked as 0):
[[0 0 0 0 3 4]
 [4 5 3 0 0 4]
 [0 0 5 4 5 0]
 [0 0 4 5 4 0]
 [3 0 3 0 3 3]]

Test entries: 4

Test Matrix (only test entries shown):
[[5 4 0 0 0 0]
 [0 0 0 0 0 0]
 [0 0 0 0 0 0]
 [0 0 0 0 0 0]
 [0 3 0 3 0 0]]


## 7. End-to-End Evaluation & Results

Now we'll train both models, make predictions, and evaluate their performance.


In [ ]:
# Train User-based CF
print("=" * 60)
print("USER-BASED COLLABORATIVE FILTERING")
print("=" * 60)

user_cf = UserBasedCF(k=3)  # Use top-3 similar users
user_cf.fit(train_matrix)

print("\nUser-User Similarity Matrix:")
print(user_cf.user_similarity.round(3))
print("\nUser Means (for mean-centering):")
print(user_cf.user_means.round(2))

# Make predictions
user_predictions = user_cf.predict_all()
print("\nPredicted Ratings Matrix:")
print(user_predictions.round(2))


USER-BASED COLLABORATIVE FILTERING

User-User Similarity Matrix:
[[ 1.     0.39   0.548  0.511  0.704]
 [ 0.39   1.    -0.077 -0.121  0.364]
 [ 0.548 -0.077  1.     0.978  0.464]
 [ 0.511 -0.121  0.978  1.     0.343]
 [ 0.704  0.364  0.464  0.343  1.   ]]

User Means (for mean-centering):
[3.5  4.   4.67 4.33 3.  ]

Predicted Ratings Matrix:
[[3.5  3.5  3.51 3.48 3.51 3.5 ]
 [4.   4.   3.94 4.67 3.73 4.26]
 [4.67 4.67 4.44 5.   4.37 4.94]
 [4.33 4.33 4.58 3.67 4.37 4.63]
 [3.   4.   2.75 2.33 2.83 3.33]]


In [ ]:
# Train Item-based CF
print("=" * 60)
print("ITEM-BASED COLLABORATIVE FILTERING")
print("=" * 60)

item_cf = ItemBasedCF(k=3)  # Use top-3 similar items
item_cf.fit(train_matrix)

print("\nItem-Item Similarity Matrix:")
print(item_cf.item_similarity.round(3))
print("\nItem Means (for mean-centering):")
print(item_cf.item_means.round(2))

# Make predictions
item_predictions = item_cf.predict_all()
print("\nPredicted Ratings Matrix:")
print(item_predictions.round(2))


ITEM-BASED COLLABORATIVE FILTERING

Item-Item Similarity Matrix:
[[ 1.     0.901  0.315  0.33  -0.165  0.78 ]
 [ 0.901  1.     0.366  0.575 -0.     0.73 ]
 [ 0.315  0.366  1.     0.721  0.465 -0.302]
 [ 0.33   0.575  0.721  1.     0.721 -0.   ]
 [-0.165 -0.     0.465  0.721  1.    -0.302]
 [ 0.78   0.73  -0.302 -0.    -0.302  1.   ]]

Item Means (for mean-centering):
[3.5  5.   3.75 4.5  3.75 3.67]

Predicted Ratings Matrix:
[[3.83 5.   3.   3.75 3.75 3.67]
 [3.65 5.   3.75 4.08 3.   3.92]
 [3.   4.5  3.94 5.   3.94 4.17]
 [4.   5.   4.15 4.75 4.15 3.17]
 [2.83 4.43 3.   3.75 3.   3.17]]


In [ ]:
# Evaluate on test set
print("=" * 60)
print("EVALUATION RESULTS")
print("=" * 60)

# Extract true and predicted ratings for test set
true_ratings = ratings_full[test_mask]
user_pred_ratings = user_predictions[test_mask]
item_pred_ratings = item_predictions[test_mask]

print(f"\nTest set size: {len(true_ratings)} ratings")
print(f"True ratings: {true_ratings}")
print(f"User-CF predictions: {user_pred_ratings.round(2)}")
print(f"Item-CF predictions: {item_pred_ratings.round(2)}")

# Compute RMSE and MAE
user_rmse = rmse(true_ratings, user_pred_ratings)
user_mae = mae(true_ratings, user_pred_ratings)
item_rmse = rmse(true_ratings, item_pred_ratings)
item_mae = mae(true_ratings, item_pred_ratings)

print("\n" + "-" * 60)
print("RATING PREDICTION METRICS")
print("-" * 60)
print(f"User-based CF - RMSE: {user_rmse:.3f}, MAE: {user_mae:.3f}")
print(f"Item-based CF - RMSE: {item_rmse:.3f}, MAE: {item_mae:.3f}")

# Compute Precision@K and Recall@K for each user
k = 3
user_precisions = []
user_recalls = []
item_precisions = []
item_recalls = []

for u in range(n_users):
    user_true = ratings_full[u, :]
    user_pred_user_cf = user_predictions[u, :]
    user_pred_item_cf = item_predictions[u, :]
    
    # Mask out items not in test set for this user
    user_test_mask = test_mask[u, :]
    if np.sum(user_test_mask) > 0:
        user_prec = precision_at_k(user_true, user_pred_user_cf, k)
        user_rec = recall_at_k(user_true, user_pred_user_cf, k)
        item_prec = precision_at_k(user_true, user_pred_item_cf, k)
        item_rec = recall_at_k(user_true, user_pred_item_cf, k)
        
        user_precisions.append(user_prec)
        user_recalls.append(user_rec)
        item_precisions.append(item_prec)
        item_recalls.append(item_rec)

print("\n" + "-" * 60)
print("RANKING METRICS (Precision@3, Recall@3)")
print("-" * 60)
print(f"User-based CF - Precision@3: {np.mean(user_precisions):.3f}, Recall@3: {np.mean(user_recalls):.3f}")
print(f"Item-based CF - Precision@3: {np.mean(item_precisions):.3f}, Recall@3: {np.mean(item_recalls):.3f}")


EVALUATION RESULTS

Test set size: 4 ratings
True ratings: [5 4 3 3]
User-CF predictions: [3.5  3.5  4.   2.33]
Item-CF predictions: [3.83 5.   4.43 3.75]

------------------------------------------------------------
RATING PREDICTION METRICS
------------------------------------------------------------
User-based CF - RMSE: 0.993, MAE: 0.917
Item-based CF - RMSE: 1.113, MAE: 1.086

------------------------------------------------------------
RANKING METRICS (Precision@3, Recall@3)
------------------------------------------------------------
User-based CF - Precision@3: 0.833, Recall@3: 0.500
Item-based CF - Precision@3: 1.000, Recall@3: 0.625


In [ ]:
# Interpretation
print("\n" + "=" * 60)
print("INTERPRETATION")
print("=" * 60)
print("""
Key Observations:

1. **Similarity Patterns:**
   - Users 0 and 1 are similar (both like action/sci-fi items 0, 1, 5)
   - Users 2 and 3 are similar (both like romance/comedy items 2, 3, 4)
   - User 4 is neutral and doesn't strongly correlate with others

2. **Prediction Quality:**
   - Both methods handle sparse data by leveraging similarity
   - Mean-centering helps account for user/item rating biases
   - Cold start cases (no similar users/items) fall back to means

3. **Model Comparison:**
   - Item-based CF often more stable (items change less than users)
   - User-based CF can be more personalized but less scalable
   - Performance depends on data characteristics and sparsity

4. **Limitations:**
   - Small dataset may not show clear winner
   - Real-world systems need much larger datasets
   - Both methods struggle with cold start (new users/items)
""")



INTERPRETATION

Key Observations:

1. **Similarity Patterns:**
   - Users 0 and 1 are similar (both like action/sci-fi items 0, 1, 5)
   - Users 2 and 3 are similar (both like romance/comedy items 2, 3, 4)
   - User 4 is neutral and doesn't strongly correlate with others

2. **Prediction Quality:**
   - Both methods handle sparse data by leveraging similarity
   - Mean-centering helps account for user/item rating biases
   - Cold start cases (no similar users/items) fall back to means

3. **Model Comparison:**
   - Item-based CF often more stable (items change less than users)
   - User-based CF can be more personalized but less scalable
   - Performance depends on data characteristics and sparsity

4. **Limitations:**
   - Small dataset may not show clear winner
   - Real-world systems need much larger datasets
   - Both methods struggle with cold start (new users/items)



## 8. Interview Discussion & Extensions

### Why Item-based CF Scales Better

**Computational Complexity:**
- User-based: O(n_users² · n_items) similarity computation
- Item-based: O(n_items² · n_users) similarity computation
- Typically: n_items << n_users (e.g., 10K items vs 1M users)
- **Result**: Item-based CF is often 100x faster to compute

**Stability:**
- User preferences change frequently
- Item characteristics are more stable
- Item-item similarities can be precomputed and cached

### Memory vs Compute Tradeoffs

**Memory-efficient approach:**
- Compute similarities on-the-fly (slower, less memory)
- Use sparse matrices for storage
- Approximate nearest neighbors (LSH, ANN)

**Compute-efficient approach:**
- Precompute full similarity matrices (faster queries, more memory)
- Cache top-K neighbors for each user/item
- Use matrix factorization to reduce dimensionality

### Cold Start Mitigation Strategies

1. **New Users:**
   - Use popularity-based recommendations (most-rated items)
   - Ask for explicit preferences (onboarding)
   - Use demographic/contextual features
   - Hybrid: Content-based + CF

2. **New Items:**
   - Content-based filtering (item features)
   - Use item metadata (genre, tags, description)
   - Promote new items to diverse user segments
   - Wait for initial ratings to accumulate

### Popularity Bias

**Problem:** CF tends to recommend popular items, creating feedback loops.

**Solutions:**
- Inverse frequency weighting
- Diversity metrics (intra-list diversity)
- Serendipity measures
- Long-tail promotion

### Extensions to Discuss

#### 1. Matrix Factorization (MF)

**Idea:** Factorize R ≈ U · Vᵀ where:
- R: (n_users, n_items) rating matrix
- U: (n_users, k) user embeddings
- V: (n_items, k) item embeddings
- k: latent dimension (typically 50-200)

**Advantages:**
- Handles sparsity better
- Captures latent factors
- Scales to millions of users/items
- Basis for neural recommenders

**Algorithms:**
- SVD (Singular Value Decomposition)
- NMF (Non-negative Matrix Factorization)
- ALS (Alternating Least Squares)

#### 2. Approximate Nearest Neighbor (ANN) Search

**Problem:** Finding top-K similar users/items is O(n) for each query.

**Solutions:**
- **LSH (Locality Sensitive Hashing)**: Hash similar vectors to same buckets
- **FAISS**: Facebook's library for efficient similarity search
- **Annoy**: Spotify's approximate nearest neighbor library
- **HNSW**: Hierarchical Navigable Small World graphs

**Tradeoff:** Accuracy vs speed (99% accuracy, 100x faster)

#### 3. Hybrid Recommenders

Combine multiple signals:

**Content-based + CF:**
- Use item features when CF data is sparse
- Use CF when content features are weak

**Deep Learning:**
- Neural Collaborative Filtering (NCF)
- Wide & Deep Learning
- DeepFM, xDeepFM

**Real-time Updates:**
- Incremental learning
- Online matrix factorization
- Streaming CF

### Interview Questions to Prepare For

1. **"How would you scale this to 100M users?"**
   - Sharding, distributed computation
   - Approximate algorithms (ANN)
   - Matrix factorization
   - Incremental updates

2. **"How do you handle implicit feedback?"**
   - Convert to binary (clicked/not clicked)
   - Use ranking metrics (Precision@K, NDCG)
   - Weighted matrix factorization
   - BPR (Bayesian Personalized Ranking)

3. **"What if similarity computation is too slow?"**
   - Precompute and cache
   - Use sampling (random subset of neighbors)
   - Dimensionality reduction (PCA, SVD)
   - Approximate nearest neighbors

4. **"How do you evaluate offline vs online?"**
   - Offline: Historical train/test split, cross-validation
   - Online: A/B testing, interleaving
   - Metrics: Click-through rate, conversion, engagement

5. **"Can you modify this to use matrix factorization?"**
   - Replace similarity computation with gradient descent
   - Implement ALS or SGD optimizer
   - Add regularization (L2, dropout)
   - Handle implicit feedback

### Code Modifications for Interview

Be ready to modify the code live:

- **Change similarity metric**: Pearson correlation, Jaccard
- **Add regularization**: Shrinkage in similarity computation
- **Handle implicit feedback**: Binary matrix, ranking loss
- **Optimize for speed**: Vectorize prediction loops
- **Add cold start handling**: Fallback to popularity/means
